# 1. Find the broadcasts

Lists the interpreted news broadcasts of the three archives in `work/broadcasts.json`:

- **PrimaTV**: the videos of the FOCUS 22 & 23 playlist on PrimaTV's YouTube channel whose
  thumbnail shows the interpreter. The thumbnails are checked by eye.
- **ProTV**: the morning news (Știrile PRO TV, 06:00), from protv.ro's archive and from the
  432p files of vid2.stirileprotv.ro, by date.
- **Digi24**: the 11:00 news journal (Jurnal ora 11.00) of digi24.ro's archive, by date, with
  the 720p video file of each page.

The lists of the original run are in `frozen/broadcasts.json` (PrimaTV, Digi24). Its ProTV
lists were not kept; `frozen/alignment.json` has the protv.ro episode each ProTV broadcast was
matched to.

These are public servers: the requests are spaced (`RATE` per second).

In [ ]:
import re
import sys
import threading
import time
from datetime import date

import requests
from bs4 import BeautifulSoup

from common import WORK, read, write

RATE = 10                  # requests per second, at most
COOKIES = 'cookies.txt'    # Netscape cookies.txt of a logged-in YouTube session (PrimaTV)
OUT = WORK / 'broadcasts.json'
broadcasts = read(OUT) if OUT.exists() else {}

_lock, _last = threading.Lock(), [0.0]


def get(session, url, method='get', **kw):
    """A request, at most RATE per second (the response, whatever its status)."""
    with _lock:
        wait = _last[0] + 1 / RATE - time.monotonic()
        if wait > 0:
            time.sleep(wait)
        _last[0] = time.monotonic()
    return session.request(method, url, timeout=30, headers={'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:140.0) Gecko/20100101 Firefox/140.0'}, **kw)


MONTHS = {m: i for i, m in enumerate('ianuarie februarie martie aprilie mai iunie iulie august septembrie '
                                     'octombrie noiembrie decembrie'.split(), 1)}


def romanian_date(text):
    """'5 martie 2024' (anywhere in text) -> '2024-03-05'."""
    m = re.search(r'(\d{1,2})\s+([a-zăâîșț]+)\s+(\d{4})', text.lower())
    return date(int(m[3]), MONTHS[m[2]], int(m[1])).isoformat() if m and m[2] in MONTHS else None

## PrimaTV

The playlist's videos titled FOCUS 22 or FOCUS 23 (the evening news), and the thumbnail of each.

In [ ]:
from yt_dlp import YoutubeDL

PLAYLIST = 'https://www.youtube.com/playlist?list=PL4GqKMm44AA10WhcE1-ZFdZjhxXLXHHhU'
THUMBS = WORK / 'thumbs'
THUMBS.mkdir(parents=True, exist_ok=True)

with YoutubeDL({'quiet': True, 'skip_download': True, 'extract_flat': 'in_playlist', 'cookiefile': COOKIES}) as ydl:
    entries = ydl.extract_info(PLAYLIST, download=False)['entries']
videos = [e['id'] for e in entries if e and re.match(r'^FOCUS\s*2[23]', e.get('title') or '', re.I)]

session = requests.Session()
for vid in videos:
    path = THUMBS / f'{vid}.jpg'
    for name in ('maxresdefault', 'hqdefault'):
        if path.exists():
            break
        r = get(session, f'https://i.ytimg.com/vi/{vid}/{name}.jpg')
        if r.ok:
            path.write_bytes(r.content)
print(len(videos), 'videos,', len(list(THUMBS.glob('*.jpg'))), 'thumbnails')

Look at the interpreter box of every thumbnail and delete the ones without the interpreter:
`d` next, `a` previous, `x` delete, `q` done. The videos whose thumbnail is left are kept.

In [ ]:
import cv2

# the interpreter box in the thumbnails, by thumbnail height
THUMB_BOX = {720: (1020, 480, 221, 237), 360: (388, 225, 82, 88)}

files, i = sorted(THUMBS.glob('*.jpg')), 0
cv2.namedWindow('thumbnail', cv2.WINDOW_NORMAL)
while files:
    img = cv2.imread(str(files[i]))
    x, y, w, h = THUMB_BOX[img.shape[0]]
    print(files[i].stem)
    cv2.imshow('thumbnail', img[y:y + h, x:x + w])
    key = cv2.waitKey(0)
    if key == ord('q'):
        break
    if key == ord('x'):
        files.pop(i).unlink()
        i %= max(len(files), 1)
    elif key in (ord('d'), ord('a')):
        i = (i + (1 if key == ord('d') else -1)) % len(files)
cv2.destroyAllWindows()

broadcasts['PrimaTV'] = sorted(p.stem for p in THUMBS.glob('*.jpg'))
write(OUT, broadcasts)

## ProTV

Two routes, as in the original run:

- **protv.ro** lists its episodes newest first, about a dozen per page; the morning news are
  titled "Stirile PRO TV # 06.00 – <date>". The original run took the broadcasts up to
  2024-02-29 from here (720p).
- **vid2.stirileprotv.ro** has the 432p files of the morning news listed on stirileprotv.ro,
  at `<date>/<id>-2.mp4`, where the date is the day the file was stored, which can differ from
  the day it aired: a missing file is looked for up to two years either side. The original run
  took the broadcasts from 2024-03-01 on from here. vid2 no longer serves its files (it answers
  HTTP 500), and stirileprotv.ro now lists only the last few months; then nothing is recorded
  for vid2.

In [ ]:
LISTING = 'https://www.protv.ro/emisiuni/stirile-pro-tv/videoclipuri/episoade/pagina-'
MORNING = re.compile(r'^\s*Stirile PRO TV\s*#\s*06[.:]00')
VID2_FROM = '2024-03-01'

session, protv, page = requests.Session(), {}, 1
while True:
    r = get(session, LISTING + str(page))
    if r.status_code != 200:
        break
    soup = BeautifulSoup(r.text, 'html.parser')
    links = [a for a in soup.find_all('a', href=True) if 'episodul' in a['href']]
    if not links:
        break
    for a in links:
        title = a.get_text().strip()
        day = MORNING.match(title) and romanian_date(title)
        if day and a['href'] not in protv.setdefault(day, []):
            protv[day].append(a['href'])
    page += 1
print(page - 1, 'protv.ro pages,', len(protv), 'days')

In [ ]:
from datetime import timedelta

vid2, page = {}, 1
listed = []
while True:
    r = get(session, f'https://stirileprotv.ro/video/stirile-diminetii/?page={page}')
    items = BeautifulSoup(r.text, 'html.parser').select('div.vid_item') if r.ok else []
    for item in items:
        a = item.find('a', href=True)
        m = re.search(r'(\d{2})\.(\d{2})\.(\d{4})', a.find(class_='title').get_text(strip=True))
        if m:
            listed.append((date(int(m[3]), int(m[2]), int(m[1])), re.search(r'\d+', a['href'])[0]))
    if not items or not listed or listed[-1][0].isoformat() < VID2_FROM:
        break
    page += 1

for aired, number in listed:
    if aired.isoformat() < VID2_FROM:
        continue
    for days in [0] + [d for i in range(1, 2 * 365 + 1) for d in (-i, i)]:
        stored = aired + timedelta(days)
        url = f"https://vid2.stirileprotv.ro/{stored.strftime('%Y/%m/%d')}/{number}-2.mp4"
        status = get(session, url, 'head').status_code
        if status == 200:
            vid2.setdefault(stored.isoformat(), []).append(url)
            break
        if status != 404:          # not "no such file": the server does not serve files
            break
    if status not in (200, 404):
        print('vid2 answers', status, '- not used')
        break
broadcasts['ProTV'] = {'protv.ro': dict(sorted(protv.items())), 'vid2': dict(sorted(vid2.items()))}
write(OUT, broadcasts)
print(len(listed), 'stirileprotv.ro broadcasts,', sum(map(len, vid2.values())), 'vid2 files')

## Digi24

digi24.ro lists its journals newest first. The 11:00 journal's articles are titled
"Jurnal ora 11.00 ..." and dated "Data publicării - <date>"; a date can have more than one.
Each article page carries its player's configuration, with the video file of every rendition.

In [ ]:
import json

DIGI = 'https://www.digi24.ro'
PLAYER_CFG = re.compile(r'<script[^>]+id="video-player-cfg"[^>]*>(.*?)</script>', re.S)

session, listing, page = requests.Session(), {}, 1
while True:
    r = get(session, f'{DIGI}/emisiuni/jurnale?p={page}', allow_redirects=False)
    if r.status_code != 200:          # past the last page the site redirects
        break
    for article in BeautifulSoup(r.text, 'html.parser').find_all('article'):
        a, em = article.select_one('h2 a'), article.find('em')
        if a and em and a.get_text(strip=True).startswith('Jurnal ora 11.00') \
                and em.get_text(strip=True).startswith('Data publicării - '):
            url = DIGI + a['href']
            day = romanian_date(em.get_text())
            if url not in listing.setdefault(day, []):
                listing[day].append(url)
    page += 1

video_src = {}
for urls in listing.values():
    for url in urls:
        cfg = PLAYER_CFG.search(get(session, url).text)
        versions = json.loads(cfg[1])['new-info']['meta'].get('versions') or {} if cfg else {}
        video_src[url] = versions.get('720p.mp4')
broadcasts['Digi24'] = {'listing': dict(sorted(listing.items())), 'video_src': video_src}
write(OUT, broadcasts)
print(len(listing), 'days,', sum(v is None for v in video_src.values()), 'pages without a 720p video')